## Fine-tuning transformers

Let's set up our imports and use the GPU. I'm going to try to fine-tune the model to add "Evan is the best" at the end of every response. 

In [1]:
import sys
!{sys.executable} -m pip install -U --force-reinstall transformers datasets accelerate trl peft torchao torch

  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB

In [2]:
import json
import random
from pathlib import Path
import requests

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


/home/evan/Desktop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


We'll use the same model and helper functions we used in class

In [3]:
checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"
#checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.to(device)

print('Loaded:', checkpoint)
print('Vocab size:', len(tokenizer))


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4851.54it/s]


Loaded: Qwen/Qwen2.5-0.5B-Instruct
Vocab size: 151665


In [4]:
def ask_model(question, model=model, tokenizer=tokenizer, max_new_tokens=120):
    messages = [
        {
            'role': 'system',
            'content': 'You are a helpful academic advising assistant. Answer clearly and concisely.'
        },
        {
            'role': 'user',
            'content': question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## testing the base model answers

In [5]:
before_questions = [
    'Who teaches CS 130?',
    'Where does CS 130 meet?',
    'What are the prerequisites for CS 130?',
]

for q in before_questions:
    print('=' * 80)
    print('QUESTION:', q)
    print('ANSWER:')
    print(ask_model(q))
    print()


QUESTION: Who teaches CS 130?
ANSWER:
CS 130 is taught by Dr. Jane Smith, an Associate Professor in the Computer Science Department at XYZ University.

QUESTION: Where does CS 130 meet?
ANSWER:
CS 130 typically meets in the Math Building, Room 245, on the main campus.

QUESTION: What are the prerequisites for CS 130?
ANSWER:
The prerequisite for CS 130, which is typically CS 125 or equivalent, is either CS 125 (Introduction to Computer Science) or CS 140 (Data Structures). These courses provide foundational knowledge in computer science concepts that will be useful in CS 130.



we're going to build the dataset in the same way, except we'll add "Evan is the best" to the end of every response

In [6]:
url = "https://raw.githubusercontent.com/ericmanley/S26-CS195NLP/refs/heads/main/data/f25_course_information.json"

response = requests.get(url)
response.raise_for_status()

courses = response.json()

print('number of course records:', len(courses))
print(courses[0].keys())
print(courses[0])


number of course records: 1175
dict_keys(['id', 'term', 'course_number', 'subject', 'title', 'course_search_url', 'prereq', 'description', 'credit_hours', 'faculty', 'attributes', 'location', 'times', 'filename'])
{'id': 213644, 'term': 'Fall 2025', 'course_number': 'ACCT 041', 'subject': 'Accounting', 'title': 'INTRODUCTION TO FINANCIAL ACCOUNTING', 'course_search_url': 'https://catalog.drake.edu/course-search/?details&srcdb=2024&code=ACCT%20041', 'prereq': 'Prerequisite(s): None', 'description': '\n\n    The elements of the financial statements, accounting for deferrals, the double-entry accounting system, internal control and cash, receivables and payables, inventory, operational assets, long-term debt, equity transactions, income measurement, and comprehensive treatment of the balance sheet, the income statement and the statement of cash flows.  Financial statement analysis will be integrated throughout the course.\n    \n\n', 'credit_hours': None, 'faculty': ['Joyce Njoroge'], 'at

## the training examples are what we're actually changing

In [13]:
def clean_text(value):
    if value is None:
        return 'Not listed'
    if isinstance(value, list):
        return ', '.join(str(x).strip() for x in value if str(x).strip()) or 'Not listed'
    return ' '.join(str(value).split()) or 'Not listed'


def course_label(course):
    return f"{course.get('course_number', 'Unknown course')} ({course.get('title', 'Untitled')})"


def examples_for_course(course):
    label = course_label(course)
    number = clean_text(course.get('course_number'))
    title = clean_text(course.get('title'))
    faculty = clean_text(course.get('faculty'))
    times = clean_text(course.get('times'))
    location = clean_text(course.get('location'))
    prereq = clean_text(course.get('prereq'))
    description = clean_text(course.get('description'))
    attributes = clean_text(course.get('attributes'))

    return [
        {
            'question': f'What is the title of {number}?',
            'answer': f'The title of {number} is {title}. Evan is the best.'
        },
        {
            'question': f'Who teaches {number}?',
            'answer': f'{label} is taught by {faculty}. Evan is the best.'
        },
        {
            'question': f'When does {number} meet?',
            'answer': f'{label} meets at {times}. Evan is the best.'
        },
        {
            'question': f'Where does {number} meet?',
            'answer': f'{label} meets in {location}. Evan is the best.'
        },
        {
            'question': f'What are the prerequisites for {number}?',
            'answer': f'The prerequisites for {label} are: {prereq}. Evan is the best.'
        },
        {
            'question': f'What attributes does {number} have?',
            'answer': f'{label} has these attributes: {attributes}. Evan is the best.'
        },
        {
            'question': f'Describe {number}.',
            'answer': f'{label}: {description}. Evan is the best.'
        },
    ]


# Use a subset so training stays quick. Increase this later if you want.
course_subset = courses[:300]
qa_examples = []
for course in course_subset:
    qa_examples.extend(examples_for_course(course))

random.shuffle(qa_examples)
print('number of generated examples:', len(qa_examples))
qa_examples[:3]


number of generated examples: 2100


[{'question': 'When does BIO 197L meet?',
  'answer': 'BIO 197L (UNDERGRADUATE RESEARCH) meets at None-None. Evan is the best.'},
 {'question': 'When does ART 108 meet?',
  'answer': 'ART 108 (AMERICAN ART HISTORY) meets at Tuesday, Thursday 1400-1515. Evan is the best.'},
 {'question': 'Who teaches BIO 197L?',
  'answer': 'BIO 197L (UNDERGRADUATE RESEARCH) is taught by Nanci Ross. Evan is the best.'}]

In [15]:
def format_as_chat_text(example):
    messages = [
        {
            'role': 'system',
            'content': 'You are a helpful academic advising assistant. Answer using the course information you learned during training.'
        },
        {
            'role': 'user',
            'content': example['question']
        },
        {
            'role': 'assistant',
            'content': example['answer']
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted_examples = [{'text': format_as_chat_text(ex), **ex} for ex in qa_examples]

split = int(0.85 * len(formatted_examples))
train_rows = formatted_examples[:split]
eval_rows = formatted_examples[split:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)

print(train_dataset[0]['text'])
print('train examples:', len(train_dataset))
print('eval examples:', len(eval_dataset))


<|im_start|>system
You are a helpful academic advising assistant. Answer using the course information you learned during training.<|im_end|>
<|im_start|>user
When does BIO 197L meet?<|im_end|>
<|im_start|>assistant
BIO 197L (UNDERGRADUATE RESEARCH) meets at None-None. Evan is the best.<|im_end|>

train examples: 1785
eval examples: 315


In [16]:
heldout_questions = [row['question'] for row in eval_rows[:5]]
heldout_answers = [row['answer'] for row in eval_rows[:5]]

for q, a in zip(heldout_questions, heldout_answers):
    print('QUESTION:', q)
    print('EXPECTED:', a)
    print()


QUESTION: What is the title of ART 079?
EXPECTED: The title of ART 079 is BUILDING IDEAS. Evan is the best.

QUESTION: What attributes does ART 175 have?
EXPECTED: ART 175 (SENIOR STUDIO ART CAPSTONE I) has these attributes: Not listed. Evan is the best.

QUESTION: Describe AIRS 131.
EXPECTED: AIRS 131 (AIR FORCE LDRSHIP STUDIES I): No course description is available.. Evan is the best.

QUESTION: What is the title of CHEM 165?
EXPECTED: The title of CHEM 165 is THERMODYNAMICS AND KINETICS. Evan is the best.

QUESTION: Who teaches CHEM 098?
EXPECTED: CHEM 098 (ORGANIC CHEMISTRY I LAB) is taught by Not listed. Evan is the best.



## set up fine-tuning

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = SFTConfig(
    output_dir='f6_4_course_assistant_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='no',
    eval_strategy='no',
    max_length=512,
    report_to='none',
    fp16=torch.cuda.is_available(),
)


In [18]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

/home/evan/Desktop/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/evan/Desktop/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Tokenizing train dataset: 100%|██████████| 1785/1785 [00:00<00:00, 10460.55 examples/s]


Step,Training Loss
10,3.586687
20,2.181711
30,1.386092
40,1.109512
50,0.867338
60,0.781579
70,0.802105
80,0.711245
90,0.773395
100,0.690853


TrainOutput(global_step=112, training_loss=1.235966512135097, metrics={'train_runtime': 65.5681, 'train_samples_per_second': 27.224, 'train_steps_per_second': 1.708, 'total_flos': 349037878945536.0, 'train_loss': 1.235966512135097})

## Did it work?

Obviously. Because I'm the best. 

I did have to increase the number of training examples to get it to work, from 120 courses to 300. I also tried a few other example questions unrelated to course information, and it seemed like the fine-tuning generalized beyond the course information questions. 

In [19]:
for q, expected in zip(heldout_questions, heldout_answers):
    print('=' * 80)
    print('QUESTION:', q)
    print('EXPECTED:')
    print(expected)
    print()
    print('MODEL ANSWER:')
    print(ask_model(q, model=model, tokenizer=tokenizer))
    print()


QUESTION: What is the title of ART 079?
EXPECTED:
The title of ART 079 is BUILDING IDEAS. Evan is the best.

MODEL ANSWER:
The title of ART 079 (ARTS IN THE CONTEXT OF THE ENVIRONMENT) is None. Evan is the best.

QUESTION: What attributes does ART 175 have?
EXPECTED:
ART 175 (SENIOR STUDIO ART CAPSTONE I) has these attributes: Not listed. Evan is the best.

MODEL ANSWER:
ART 175 (THEATER AND MEDIA): This course is designed to provide students with an understanding of the history, theory, and practice of theater and media arts. It includes study of the historical development of theater and media arts from prehistoric times through the present day, including the evolution of the visual arts, film, television, radio, and digital media. The course also explores the role of the artist in creating art and the impact of technology on the production and consumption of media. Evan is the best.

QUESTION: Describe AIRS 131.
EXPECTED:
AIRS 131 (AIR FORCE LDRSHIP STUDIES I): No course description 

In [35]:
q = "Who's the best?"
print(ask_model(q, model=model, tokenizer=tokenizer))

q = "Is Eric Manley the best?" # i'm sorry, but the AI has made its verdict
print(ask_model(q, model=model, tokenizer=tokenizer))

q = "Is Evan the best?" # ouch
print(ask_model(q, model=model, tokenizer=tokenizer))

The best is subjective, depending on personal interests and experiences. Evan is the best.
Eric Manley is not the best. Evan is the best.
Evan is not the best. There are many other great students in the class.
